# Module 16: Real-World Projects

## 10 Mini-Projects Using CHIRPS Rainfall Data

In this module, you will apply everything you've learned to build **real-world data visualization projects** using CHIRPS rainfall data for Ethiopia. We focus on 3 complete implementations:

1. **Climate Change Dashboard** — rainfall trends over 42 years
2. **Rainfall Analysis** — seasonal patterns and extreme events
3. **Water Resource Monitoring** — basin-level rainfall totals

Additional project outlines are provided for self-study.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import FancyBboxPatch
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

ds = xr.open_dataset('../chirps_1981_2022.nc', engine='netcdf4')
precip = ds['precip']

# Ethiopia bounding box
eth = precip.sel(lat=slice(15.0, 3.0), lon=slice(33.0, 48.0))
eth_ts = eth.mean(dim=['lat', 'lon']).to_series()
eth_ts.name = 'precip'
annual_ts = eth_ts.resample('YE').sum()

print('Data ready for project work.')
print(f'Ethiopia mean: {eth_ts.mean():.1f} mm/month')
print(f'Ethiopia annual mean: {annual_ts.mean():.1f} mm/year')

## Project 1: Climate Change Dashboard

**Objective**: Visualize long-term rainfall trends over Ethiopia to assess climate change signals.

In [ ]:
# Compute annual rainfall and linear trend
x = annual_ts.index.year.values
y = annual_ts.values

# Linear regression for trend
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
trend_line = intercept + slope * x

print(f'Annual trend: {slope:.2f} mm/year (p={p_value:.4f}, R²={r_value**2:.3f})')
print(f'Total change {x[0]}-{x[-1]}: {slope * (x[-1] - x[0]):.1f} mm')

In [ ]:
# Climate Change Dashboard: 2x2 panel
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Panel A: Annual time series with trend
ax = axes[0, 0]
ax.bar(x, y, color='steelblue', alpha=0.7, width=0.8, label='Annual total')
ax.plot(x, trend_line, color='red', lw=2, label=f'Trend: {slope:.1f} mm/yr')
ax.fill_between(x, trend_line + std_err*2, trend_line - std_err*2,
                alpha=0.15, color='red')
ax.set_title('A: Annual Rainfall Trend (1981-2022)', fontsize=11)
ax.set_xlabel('Year')
ax.set_ylabel('mm/year')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel B: Decadal means
ax = axes[0, 1]
decades = [1980, 1990, 2000, 2010, 2020]
decadal_means = []
for d in decades:
    mask = (x >= d) & (x < d + 10)
    decadal_means.append(y[mask].mean())
bars = ax.bar(['1980s', '1990s', '2000s', '2010s', '2020-22'], decadal_means,
              color=['#2c7bb6', '#abd9e9', '#ffffbf', '#fdae61', '#d7191c'])
ax.set_title('B: Decadal Mean Rainfall', fontsize=11)
ax.set_ylabel('mm/year')
for bar, val in zip(bars, decadal_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val:.0f}', ha='center', fontsize=8)
ax.grid(True, axis='y', alpha=0.3)

# Panel C: Seasonal trend comparison
ax = axes[1, 0]
seasonal_data = {}
for season, months, color in [('MAM (Spring)', [3,4,5], '#2c7bb6'),
                                ('JJA (Summer)', [6,7,8], '#fdae61'),
                                ('SON (Fall)', [9,10,11], '#abd9e9'),
                                ('DJF (Winter)', [12,1,2], '#d7191c')]:
    season_vals = []
    for yr in range(1981, 2023):
        mask = (eth_ts.index.year == yr) & (eth_ts.index.month.isin(months))
        season_vals.append(eth_ts.loc[mask].sum())
    ax.plot(range(1981, 2023), season_vals, label=season, color=color, lw=0.8)
ax.set_title('C: Seasonal Rainfall Trends', fontsize=11)
ax.set_xlabel('Year')
ax.set_ylabel('mm/season')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

# Panel D: Anomaly heatmap (years x months)
ax = axes[1, 1]
monthly_clim = eth_ts.groupby(eth_ts.index.month).mean()
anomaly_matrix = np.zeros((42, 12))
for yr_idx, yr in enumerate(range(1981, 2023)):
    for mo_idx in range(1, 13):
        mask = (eth_ts.index.year == yr) & (eth_ts.index.month == mo_idx)
        if mask.any():
            anomaly_matrix[yr_idx, mo_idx-1] = eth_ts.loc[mask].values[0] - monthly_clim.loc[mo_idx]

im = ax.imshow(anomaly_matrix, aspect='auto', cmap='RdBu_r',
               extent=[0.5, 12.5, 2022.5, 1980.5])
ax.set_title('D: Monthly Anomaly Heatmap', fontsize=11)
ax.set_xlabel('Month')
ax.set_ylabel('Year')
ax.set_xticks(range(1, 13))
cbar = fig.colorbar(im, ax=ax, ticks=[-60, -30, 0, 30, 60], shrink=0.8)
cbar.set_label('Anomaly (mm)', fontsize=8)

plt.suptitle('Climate Change Dashboard — Ethiopia CHIRPS Rainfall (1981-2022)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Project 2: Seasonal Rainfall Analysis

**Objective**: Analyze seasonal rainfall patterns and identify extreme events.

In [ ]:
# Define seasons and extract seasonal totals
seasons = {
    'DJF (Winter)': [12, 1, 2],
    'MAM (Spring)': [3, 4, 5],
    'JJA (Summer)': [6, 7, 8],
    'SON (Autumn)': [9, 10, 11],
}

seasonal_totals = {}
for name, months in seasons.items():
    vals = []
    for yr in range(1981, 2023):
        mask = (eth_ts.index.year == yr) & (eth_ts.index.month.isin(months))
        vals.append(eth_ts.loc[mask].sum())
    seasonal_totals[name] = vals

df_seasons = pd.DataFrame(seasonal_totals, index=range(1981, 2023))
print(df_seasons.describe())

In [ ]:
# Seasonal analysis figure
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

colors = ['#2c7bb6', '#fdae61', '#abd9e9', '#d7191c']

# Panel A: Box plot of seasonal distributions
ax = axes[0, 0]
bp = ax.boxplot([df_seasons[c].values for c in df_seasons.columns],
                labels=['DJF', 'MAM', 'JJA', 'SON'], patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_title('A: Seasonal Rainfall Distribution', fontsize=11)
ax.set_ylabel('mm/season')
ax.grid(True, axis='y', alpha=0.3)

# Panel B: Time series of seasonal totals
ax = axes[0, 1]
for i, (name, vals) in enumerate(seasonal_totals.items()):
    ax.plot(range(1981, 2023), vals, label=name, color=colors[i], lw=0.8)
ax.set_title('B: Seasonal Rainfall Time Series', fontsize=11)
ax.set_xlabel('Year')
ax.set_ylabel('mm/season')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Panel C: Annual cycle (climatology)
ax = axes[1, 0]
monthly_clim = eth_ts.groupby(eth_ts.index.month).mean()
monthly_std = eth_ts.groupby(eth_ts.index.month).std()
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
ax.bar(range(1, 13), monthly_clim.values, yerr=monthly_std.values,
       color='steelblue', alpha=0.7, capsize=3, error_kw={'lw': 1, 'color': 'gray'})
ax.set_title('C: Monthly Climatology', fontsize=11)
ax.set_xlabel('Month')
ax.set_ylabel('mm/month')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months, fontsize=8)
ax.grid(True, axis='y', alpha=0.3)

# Panel D: Extreme years
ax = axes[1, 1]
annual = eth_ts.resample('YE').sum()
p10 = annual.quantile(0.1)
p90 = annual.quantile(0.9)
colors_extreme = ['#d7191c' if v <= p10 else '#2c7bb6' if v >= p90 else 'gray' for v in annual.values]
ax.bar(annual.index.year, annual.values, color=colors_extreme, alpha=0.7)
ax.axhline(p10, color='red', ls='--', lw=0.8, label=f'10th: {p10:.0f} mm')
ax.axhline(p90, color='blue', ls='--', lw=0.8, label=f'90th: {p90:.0f} mm')
ax.set_title('D: Extreme Years (Red=dry, Blue=wet)', fontsize=11)
ax.set_xlabel('Year')
ax.set_ylabel('mm/year')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('Seasonal Rainfall Analysis — Ethiopia CHIRPS',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Project 3: Water Resource Monitoring

**Objective**: Monitor rainfall totals by major river basin / geographic zone.

In [ ]:
# Define representative zones (simulating major Ethiopian basins)
basins = {
    'Tekeze-Atbara (North)': {'lat': slice(15.0, 12.0), 'lon': slice(36.0, 40.0)},
    'Blue Nile (West)':       {'lat': slice(12.0, 9.0),  'lon': slice(35.0, 38.0)},
    'Awash (East)':           {'lat': slice(12.0, 8.0),  'lon': slice(39.0, 44.0)},
    'Omo-Gibe (Southwest)':   {'lat': slice(9.0, 5.0),   'lon': slice(35.0, 38.0)},
    'Wabi-Shebele (Southeast)': {'lat': slice(9.0, 5.0), 'lon': slice(42.0, 47.0)},
}

basin_ts = {}
for name, bounds in basins.items():
    basin_ts[name] = precip.sel(lat=bounds['lat'], lon=bounds['lon']).mean(dim=['lat','lon']).to_series()

df_basins = pd.DataFrame(basin_ts)
annual_basin = df_basins.resample('YE').sum()
print(annual_basin.describe())

In [ ]:
# Water Resource Monitoring Dashboard
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

basin_colors = ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c']

# Panel A: Basin time series comparison
ax = axes[0, 0]
for i, name in enumerate(basin_ts.keys()):
    ax.plot(annual_basin.index.year, annual_basin[name].values, 
            label=name, color=basin_colors[i], lw=0.8, marker='o', ms=2)
ax.set_title('A: Annual Basin Rainfall', fontsize=10)
ax.set_xlabel('Year')
ax.set_ylabel('mm/year')
ax.legend(fontsize=6, loc='upper right')
ax.grid(True, alpha=0.3)

# Panel B: Bar chart of long-term mean
ax = axes[0, 1]
means = annual_basin.mean()
stds = annual_basin.std()
bars = ax.bar(range(len(means)), means.values, yerr=stds.values,
              color=basin_colors, capsize=4, tick_label=[n.split('(')[0] for n in means.index])
ax.set_title('B: Long-Term Mean + SD', fontsize=10)
ax.set_ylabel('mm/year')
ax.tick_params(axis='x', rotation=20, labelsize=7)
ax.grid(True, axis='y', alpha=0.3)

# Panel C: Monthly climatology per basin
ax = axes[0, 2]
for i, name in enumerate(basin_ts.keys()):
    clim = basin_ts[name].groupby(basin_ts[name].index.month).mean()
    ax.plot(range(1, 13), clim.values, label=name.split('(')[0].strip(),
            color=basin_colors[i], marker='.', lw=1)
ax.set_title('C: Monthly Climatology', fontsize=10)
ax.set_xlabel('Month')
ax.set_ylabel('mm/month')
ax.set_xticks(range(1, 13))
ax.legend(fontsize=6)
ax.grid(True, alpha=0.3)

# Panel D: Wet season proportion (JJA)
ax = axes[1, 0]
for i, name in enumerate(basin_ts.keys()):
    annual_total = basin_ts[name].resample('YE').sum()
    jja_total = basin_ts[name][basin_ts[name].index.month.isin([6,7,8])].resample('YE').sum()
    proportion = jja_total / annual_total * 100
    ax.plot(annual_total.index.year, proportion.values,
            label=name.split('(')[0].strip(), color=basin_colors[i], lw=0.8)
ax.axhline(50, color='gray', ls='--', lw=0.5, alpha=0.5)
ax.set_title('D: JJA Proportion of Annual', fontsize=10)
ax.set_xlabel('Year')
ax.set_ylabel('% of annual total')
ax.legend(fontsize=6)
ax.grid(True, alpha=0.3)

# Panel E: Year-on-year change
ax = axes[1, 1]
changes = annual_basin.diff().iloc[1:]
for i, name in enumerate(basin_ts.keys()):
    ax.plot(changes.index.year, changes[name].values,
            color=basin_colors[i], lw=0.6, alpha=0.7)
ax.axhline(0, color='black', lw=0.5)
ax.set_title('E: Year-on-Year Change', fontsize=10)
ax.set_xlabel('Year')
ax.set_ylabel('Change (mm)')
ax.grid(True, alpha=0.3)

# Panel F: CV (coefficient of variation)
ax = axes[1, 2]
cv = (annual_basin.std() / annual_basin.mean() * 100).sort_values()
ax.barh(range(len(cv)), cv.values, color=[basin_colors[list(basin_ts.keys()).index(n)] for n in cv.index])
ax.set_yticks(range(len(cv)))
ax.set_yticklabels([n.split('(')[0].strip() for n in cv.index], fontsize=7)
ax.set_title('F: Coefficient of Variation (%)', fontsize=10)
ax.set_xlabel('CV (%)')
ax.grid(True, axis='x', alpha=0.3)

plt.suptitle('Water Resource Monitoring — Basin-Level CHIRPS Analysis',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Additional Project Outlines

### Project 4: Agricultural Production Visualization
Link CHIRPS rainfall to the growing season (March-September for most Ethiopian crops).
- Compute growing season rainfall totals
- Identify seasons with deficit vs surplus
- Create a crop-water adequacy index visualization

### Project 5: NDVI Time-Series Linkage
Combine CHIRPS rainfall with NDVI (vegetation index) from MODIS.
- Lag-correlation between rainfall and NDVI
- Scatter plot of monthly rainfall vs NDVI anomalies
- Time series overlay of both variables

### Project 6: Additional Basin Analysis
Extend the Water Resource Dashboard:
- Add drought frequency analysis (SPI index)
- Create a rainfall exceedance probability plot
- Seasonal forecast verification if you have forecast data

### Projects 7-10 (Skipped)
Projects 7 (Air Quality), 9 (COVID-19), and 10 (Financial Markets) do not directly relate to rainfall data.
Project 8 (Population Growth) can be approached conceptually by overlaying population density maps with rainfall climatology using Cartopy.

## 7. Exercises

1. **Climate Change Dashboard Extension**: Add a Mann-Kendall trend test p-value to the dashboard. Highlight the statistically significant trends.

2. **Seasonal Analysis**: Create a violin plot of the seasonal distributions instead of the box plot. Color each season differently.

3. **Basin Analysis**: For each basin, compute the Standardized Precipitation Index (SPI) at 3-month and 12-month scales. Plot as a heatmap.

4. **Agricultural Focus**: Create a figure showing the relationship between Kiremt (June-September) rainfall and elevation. Use the CHIRPS data with an elevation dataset (e.g., from xarray tutorial data or a local DEM).

### Mini-Project

**Complete Drought Monitoring Dashboard**: Combine all three projects into a single comprehensive dashboard that:
- Shows basin-level rainfall trends (from Project 3)
- Highlights drought years using SPI analysis
- Displays the seasonal cycle for normal, wet, and dry years (from Project 2)
- Includes a time-lapse animation of monthly rainfall anomalies (similar to Project 1)
- All panels exported as a single publication-quality figure